**Completed experiment — do not rerun unchanged.** The uploaded 22-record run failed the release criteria. See the [audit](https://github.com/McChandra/medical_chatbot/blob/feature/responsive-healthbrowse-auth/healthbrowse/inference/releases/llama3-general-research-002-grounded-v2-evaluation.md). This notebook stays pinned to the historical code and does not include later safeguards.

# HealthBrowse: focused gateway evaluation, version 2

This tests the changes made after your completed first evaluation. **Do not retrain or rerun the old 78-generation notebook.** Your existing adapter remains unchanged.

The new run compares base Llama and the adapter on **eight browser/buddy questions**, with base-model support review and at most one repair. It also runs **six reviewer regression checks** based on the observed citation and question-coverage failures. That produces **22 result records**; each gateway case may involve several model calls.

Start a **fresh GPU runtime**, not the runtime still holding the earlier model. Use **File → Save a copy in Drive**, then run cells in order. Keep `HF_TOKEN` in Colab Secrets and grant this notebook access. The saved training folder is reused. No paid inference API or public tunnel is used.

The inference service listens only on loopback while the notebook actively evaluates it, then stops. This is not website deployment. Authentication and quotas use fixed test fixtures; actual model generation, source validation, support review, repair and the 120-second gateway deadline run. Passing automated review is not medical accuracy or clinical approval.

## 1. Install the pinned code and compatible dependencies

Setup removes optional incompatible vision/audio packages from this text-only session. If it reports cached imports, choose **Runtime → Restart session**, then run from the top. Keep all Drive files. The repository must be accessible to your Git client; do not put a GitHub token in a cell or clone URL.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os, tempfile, re
SOURCE_COMMIT = '842dab12a047dce6506cd8a99b54b3d84f1706f5'
SOURCE_URL = 'https://github.com/McChandra/medical_chatbot.git'
CACHE_ROOT = Path('/content')

def ensure_checkout(cache_root, commit, repository_url):
    if not re.fullmatch(r'[0-9a-f]{40}', commit):
        raise ValueError('Use a full source commit SHA.')
    cache_root = Path(cache_root)
    cache_root.mkdir(parents=True, exist_ok=True)
    destination = cache_root / ('healthbrowse-qlora-' + commit[:12])
    if not destination.exists():
        # Build in a new temporary directory; a failed fetch cannot leave a stale
        # final checkout. Never modify /content/medical_chatbot or saved Drive runs.
        with tempfile.TemporaryDirectory(prefix='hb-download-', dir=cache_root) as temporary:
            staging = Path(temporary) / 'repo'
            staging.mkdir()
            subprocess.run(['git','init','--quiet',str(staging)],check=True)
            subprocess.run(['git','-C',str(staging),'fetch','--quiet','--depth','1',repository_url,commit],check=True)
            subprocess.run(['git','-C',str(staging),'checkout','--quiet','--detach',commit],check=True)
            staging.rename(destination)
    if not (destination/'.git').is_dir():
        raise RuntimeError('The versioned path contains other files. Keep them and choose another CACHE_ROOT.')
    actual = subprocess.check_output(['git','-C',str(destination),'rev-parse','HEAD'],text=True).strip()
    if actual != commit:
        raise RuntimeError('The cached checkout has changed. Preserve it and choose another CACHE_ROOT.')
    if subprocess.check_output(['git','-C',str(destination),'status','--porcelain'],text=True).strip():
        raise RuntimeError('The cached checkout has local edits. Preserve them and choose another CACHE_ROOT.')
    return destination

REPO = ensure_checkout(CACHE_ROOT, SOURCE_COMMIT, SOURCE_URL)
INFERENCE = REPO / 'healthbrowse' / 'inference'
required = ['run_grounded_pipeline.py','serve.py','structured_output.py','adapter_artifact.py','requirements.txt']
missing = [name for name in required if not (INFERENCE/name).is_file()]
if missing:
    raise RuntimeError('Incomplete pinned source checkout: ' + ', '.join(missing))
SOURCE_VERSION = SOURCE_COMMIT
# A prior notebook attempt may have cached imports from the old checkout.
for name in ['evaluate_adapter','evaluation_checks','adapter_artifact','model_settings']:
    module = sys.modules.get(name)
    if module and Path(getattr(module,'__file__','')).parent != INFERENCE:
        del sys.modules[name]
print('Source version:', SOURCE_VERSION)
print('Source folder:', REPO)
print('Evaluation scripts: ready')
def prepare_text_environment(requirements):
    # This notebook evaluates text-only Llama. Colab's preinstalled image/audio
    # extensions may target a different torch ABI than the pinned torch 2.8.
    imported_before = [name for name in ['torch','transformers','torchvision','torchaudio','peft','bitsandbytes']
                       if name in sys.modules]
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(requirements)],check=True)
    # Use a fresh interpreter: imported modules in this kernel may still refer
    # to the packages that pip just replaced or removed.
    smoke = (
        "import torch; "
        "from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, "
        "GenerationConfig; "
        "from peft import PeftModel; "
        "import bitsandbytes; "
        "import lmformatenforcer, jsonschema; "
        "print('Text evaluation imports: OK; torch=' + torch.__version__)"
    )
    result = subprocess.run([sys.executable,'-c',smoke],stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT,text=True)
    print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Dependency import check failed before evaluation. Full error tail:\n' + result.stdout[-12000:])
    if imported_before:
        raise RuntimeError('Dependencies are repaired, but this session has cached imports. '
                           'Choose Runtime > Restart session (not Disconnect and delete runtime), '
                           'then run this notebook from the top. Drive files and the source checkout remain.')

prepare_text_environment(INFERENCE/'requirements.txt')

## 2. Prepare Node.js for the actual JavaScript gateway

No npm packages are needed by this evaluation runner. If the runtime does not already have Node.js 22 or newer, this cell downloads the official Node.js 22 Linux binary, checks it against the release checksum, and extracts only that binary. The actual Node version is recorded with results.

In [ ]:
import shutil, urllib.request, hashlib, tarfile, platform
node_path = shutil.which('node')
node_version = subprocess.check_output([node_path, '--version'], text=True).strip() if node_path else ''
if not node_version or int(node_version.lstrip('v').split('.')[0]) < 22:
    if platform.system() != 'Linux' or platform.machine() not in {'x86_64', 'AMD64'}:
        raise RuntimeError('This setup expects a standard Linux x64 Colab GPU runtime.')
    node_base = 'https://nodejs.org/dist/latest-v22.x/'
    with urllib.request.urlopen(node_base + 'SHASUMS256.txt', timeout=60) as response:
        checksums = response.read().decode()
    matches = re.findall(r'^([a-f0-9]{64})\s+(node-v22\.\d+\.\d+-linux-x64\.tar\.xz)$', checksums, re.M)
    if len(matches) != 1:
        raise RuntimeError('Cannot identify the official Linux x64 Node release.')
    expected, name = matches[0]
    with tempfile.TemporaryDirectory(prefix='hb-node-') as directory:
        archive = Path(directory)/name
        with urllib.request.urlopen(node_base + name, timeout=60) as response, archive.open('wb') as target:
            shutil.copyfileobj(response, target)
        if hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise RuntimeError('Node archive checksum mismatch.')
        binary_dir = Path('/content')/name.removesuffix('.tar.xz')/'bin'
        binary_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive, 'r:xz') as tar:
            member = tar.getmember(name.removesuffix('.tar.xz') + '/bin/node')
            if not member.isfile():
                raise RuntimeError('Expected a regular Node binary.')
            with tar.extractfile(member) as source, (binary_dir/'node').open('wb') as target:
                shutil.copyfileobj(source, target)
        (binary_dir/'node').chmod(0o755)
    os.environ['PATH'] = str(binary_dir) + os.pathsep + os.environ['PATH']
print(subprocess.check_output(['node', '--version'], text=True).strip())

## 3. Mount Drive and reuse the trained adapter

The default path is `MyDrive/HealthBrowse/llama3-general-research-002`. If that folder is present, no upload is needed. Only if it is missing, upload your original **Llama-3-HealthBrowse-General-QLoRA.zip**, not an evaluation ZIP. The adapter checksum is verified before use.

In [ ]:
from google.colab import userdata, drive, files
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets and enable notebook access. Do not paste it into source.') from exc
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is empty. Set it in Colab Secrets.')
drive.mount('/content/drive')
sys.path.insert(0, str(INFERENCE))
from adapter_artifact import extract_export, EXPECTED_ADAPTER_SHA256
ARTIFACT_DIR = Path('/content/drive/MyDrive/HealthBrowse/llama3-general-research-002')
if not (ARTIFACT_DIR/'adapter/adapter_model.safetensors').is_file():
    print('Saved adapter not found. Select the exported Llama-3-HealthBrowse-General-QLoRA.zip.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one exported adapter ZIP.')
    name, data = next(iter(uploaded.items()))
    if Path(name).name != name or not name.lower().endswith('.zip'):
        raise ValueError('Expected one ZIP filename without path components.')
    imported = Path('/content/drive/MyDrive/HealthBrowse/llama3-general-research-002-imported')
    with tempfile.TemporaryDirectory(prefix='hb-export-') as folder:
        archive = Path(folder)/'adapter-export.zip'
        archive.write_bytes(data)
        extract_export(archive, imported)
    ARTIFACT_DIR = imported
    del uploaded, data
print('Using saved artifact:', ARTIFACT_DIR)
print('Expected adapter SHA256:', EXPECTED_ADAPTER_SHA256)

## 4. Check the run without loading weights

Results go to a **new** `evaluation-grounded-v2-001` folder. Completed case records are saved individually. Rerun with the same name to resume the same code, GPU type and package versions. If those change, choose a new name instead of mixing comparisons. The first evaluation and training files are preserved.

In [ ]:
EVALUATION_NAME = 'evaluation-grounded-v2-001'
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,80}', EVALUATION_NAME):
    raise ValueError('Use a simple evaluation folder name.')
EVAL_DIR = ARTIFACT_DIR/EVALUATION_NAME
cmd = [sys.executable, str(INFERENCE/'run_grounded_pipeline.py'),
       '--artifact', str(ARTIFACT_DIR), '--output', str(EVAL_DIR)]
subprocess.run(cmd + ['--validate-only'], check=True)

## 5. Run the actual gateway pipeline

Model download/loading can take several minutes. The notebook displays progress and saves diagnostic logs. Each completed question is retained, even when the gateway rejects its model response. A timeout or failed support review is an evaluation finding; it must not be relabeled a successful medical answer.

The service is authenticated, uses 4-bit weights, and has no public listener. Keep the notebook open and monitor it. Free Colab availability is not guaranteed. No keep-alive workaround is used.

In [ ]:
from collections import deque
from datetime import datetime, timezone
import importlib.metadata
import uuid

def run_evaluation(command, log_dir):
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True,exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    log_path = log_dir / ('evaluation-' + stamp + '-' + uuid.uuid4().hex[:8] + '.log')
    packages = {}
    for name in ['torch','transformers','accelerate','peft','bitsandbytes','torchvision']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = 'not installed'
    tail = deque(maxlen=80)
    secrets = [os.environ[name] for name in ['HF_TOKEN','HUGGING_FACE_HUB_TOKEN'] if os.environ.get(name)]
    def redact(text):
        for value in secrets:
            text = text.replace(value,'[REDACTED]')
        return text
    print('Evaluation log:',log_path)
    with log_path.open('w',encoding='utf-8') as log:
        log.write(json.dumps({'python':sys.version,'packages':packages})+'\n')
        log.flush()
        process = subprocess.Popen([command[0],'-u',*command[1:]],stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        try:
            for line in process.stdout:
                line = redact(line)
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if returncode:
        details = ''.join(tail)[-12000:] or '(The process produced no output.)'
        raise RuntimeError(f'Evaluation exited with code {returncode}. Log: {log_path}\n'
                           'The actual evaluation output is below. Share the traceback, not just the exit code.\n'
                           + details)
    return log_path



EVALUATION_LOG = run_evaluation(cmd, EVAL_DIR/'logs')
if not (EVAL_DIR/'summary.json').is_file():
    raise RuntimeError('No summary was produced. Inspect the saved evaluation log.')
print(json.dumps(json.loads((EVAL_DIR/'summary.json').read_text()), indent=2))

## 6. Download the new result ZIP

The ZIP includes raw model requests/responses for the fixed public/synthetic benchmark, the final gateway responses, timing, runtime versions and source fixture checksums. It excludes model weights, tokens and service logs. No visitor records or real patient information are used.

Upload **HealthBrowse-Grounded-V2-Evaluation.zip** in the conversation. We will inspect false acceptance of unsupported claims, question gaps, actual review/repair behavior and latency before deciding whether either model route is ready for a private pilot. Automated acceptance is not clinical certification.

In [ ]:
import zipfile
archive = EVAL_DIR/'HealthBrowse-Grounded-V2-Evaluation.zip'
required = ['evaluation-manifest.json', 'runtime.json', 'results.jsonl', 'summary.json']
for name in required:
    if not (EVAL_DIR/name).is_file():
        raise RuntimeError('Missing output: ' + name)
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for name in required:
        zipped.write(EVAL_DIR/name, name)
    for name in ['cases.json', 'manifest.json']:
        zipped.write(INFERENCE/'evaluation-v2'/name, 'evaluation-fixtures/'+name)
files.download(str(archive))

References: [LM Format Enforcer](https://github.com/noamgat/lm-format-enforcer), [Meta Llama 3 model and terms](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct), [Colab FAQ](https://research.google.com/colaboratory/faq.html). Evidence additions retain CDC/NCI attribution and source URLs in the fixtures. Built with Meta Llama 3. Research only; no clinical approval is asserted.